# Transcript Intelligence: pipeline walkthrough

This notebook is the technical companion to the local dashboard. It runs the rules-only analysis on the supplied Aegis sample transcripts, then inspects the outputs that power the UI. The notebook intentionally calls the same pipeline modules as the application rather than maintaining a second implementation.

**Run from the repository's backend Python environment.** The pipeline reads `dataset/` and writes the deterministic demo artifacts to `data/processed/`. This walkthrough skips optional LLM refinement so that results are reproducible and no API key is needed.

The corpus is a fictional sample. Sentiment values are polarity scores derived from existing transcript annotations; risk and VOC outputs are heuristic review aids, not calibrated predictions or confirmed business outcomes.

## 1. Setup and run the pipeline

Call type is assigned by explainable rules over attendee domains and title signals. Product modules and themes use a fixed keyword taxonomy, weighted toward title matches. Duration-weighted utterance sentiment is computed separately for customer and Aegis speakers. OpenAI theme refinement is optional and is deliberately disabled in this notebook.

In [ ]:
from pathlib import Path
import json
import sys
from IPython.display import display

ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / "dataset").is_dir()),
    None,
)
assert ROOT is not None, "Open this notebook from within the repository."

BACKEND = ROOT / "src" / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

from pipeline.run import run_pipeline

print(f"Repository: {ROOT}")
print(f"Dataset: {ROOT / 'dataset'}")
print(f"Processed outputs: {ROOT / 'data' / 'processed'}")

In [ ]:
# Rebuild the same deterministic artifacts used by the dashboard; no model key is required.
run_result = run_pipeline(ds=ROOT / "dataset", out=ROOT / "data" / "processed", use_llm=False)
run_result

## 2. Inspect corpus coverage and call-type rules

The classifier first identifies meetings with no non-Aegis attendee as internal. Mixed-attendee support/escalation/urgent/incident title patterns become support; other mixed-attendee meetings become external. This order is simple to audit, but missing attendee metadata can lead to incorrect call types, so check the distribution and inspect examples rather than assuming the rules are perfect.

In [ ]:
processed = ROOT / "data" / "processed"
meetings = json.loads((processed / "meetings.json").read_text(encoding="utf-8"))
themes = json.loads((processed / "themes.json").read_text(encoding="utf-8"))
trends = json.loads((processed / "sentiment_trends.json").read_text(encoding="utf-8"))
heatmap = json.loads((processed / "sentiment_heatmap.json").read_text(encoding="utf-8"))
overview = json.loads((processed / "overview.json").read_text(encoding="utf-8"))
risk = json.loads((processed / "risk_radar.json").read_text(encoding="utf-8"))
graph = json.loads((processed / "issue_graph.json").read_text(encoding="utf-8"))
voc = json.loads((processed / "voc_backlog.json").read_text(encoding="utf-8"))

print(f"Meetings processed: {len(meetings)}")
print("Call-type mix:")
for call_type, count in sorted(overview["call_type_mix"].items()):
    print(f"  {call_type:8} {count:3}")

In [ ]:
import pandas as pd

call_type_examples = pd.DataFrame(
    [
        {
            "call_type": m["call_type"],
            "title": m["title"],
            "rule_reason": m["call_type_reason"],
        }
        for m in meetings
    ]
).groupby("call_type", group_keys=False).head(3).reset_index(drop=True)
call_type_examples

## 3. Theme taxonomy and transcript examples

Theme labels come from a fixed codebook in `src/backend/pipeline/taxonomy.py`. The scorer uses title matches as stronger evidence than supplied summary topics, which in turn outweigh transcript keyword hits. This keeps the results deterministic and lets reviewers inspect why a label was assigned. It can still overfit to vocabulary or title phrasing; examples below are candidates for human review, not a labeled evaluation set.

In [ ]:
meeting_by_id = {m["meeting_id"]: m for m in meetings}
theme_examples = []
for theme in themes:
    example_id = next(iter(theme["example_meeting_ids"]), None)
    meeting = meeting_by_id.get(example_id) if example_id else None
    theme_examples.append(
        {
            "theme": theme["label"],
            "count": theme["count"],
            "by_call_type": theme["by_call_type"],
            "example_title": meeting["title"] if meeting else "No example available",
            "example_call_type": meeting["call_type"] if meeting else None,
            "why_labelled": meeting["theme_rationale"] if meeting else theme["rationale"],
            "quote": meeting["example_quote"] if meeting else "",
        }
    )

pd.DataFrame(theme_examples)[["theme", "count", "by_call_type", "example_title", "example_call_type"]]

In [ ]:
# Read a representative title, rationale, and quote for each theme before presenting it.
for example in theme_examples:
    print(f"\n## {example['theme']} (n={example['count']})")
    print(f"Example: {example['example_title']} [{example['example_call_type']}]")
    print(f"Rationale: {example['why_labelled']}")
    print(f"Quote: {example['quote']}")

## 4. Sentiment by call type and week

Each utterance's existing `sentimentType` is mapped to -1, 0, or +1 and weighted by its duration (or one unit when duration is unavailable). Customer utterances are used for customer-side call scores when identified. Weekly trend points are means of meeting-level scores, so long meetings do not dominate the week. Internal calls have no customer-side score; the dashboard uses all-speaker polarity for their trend/heatmap cells. Always consider the sample count when interpreting a week.

In [ ]:
sentiment_by_type = (
    pd.DataFrame(meetings)
    .groupby("call_type")
    .agg(
        meetings=("meeting_id", "count"),
        mean_customer_polarity=("customer_sentiment_mean", "mean"),
        mean_all_speaker_polarity=("utterance_sentiment_mean", "mean"),
    )
    .round(3)
)
sentiment_by_type

In [ ]:
weekly_sentiment = pd.DataFrame(trends).pivot(index="week", columns="call_type", values="mean_customer")
weekly_counts = pd.DataFrame(trends).pivot(index="week", columns="call_type", values="n")
display(weekly_sentiment.round(2))
print("Meeting counts per week and call type:")
display(weekly_counts.fillna(0).astype(int))

module_sentiment = pd.DataFrame(heatmap).pivot(index="module", columns="call_type", values="mean")
print("Module × call-type mean polarity (internal values use all-speaker fallback):")
display(module_sentiment.round(2))

In [ ]:
print("Pipeline-generated interpretation:")
for takeaway in overview["takeaways"]:
    print(f"- {takeaway}")

print("\nInterpretation guardrail: these are descriptive correlations in a small, fictional sample; they do not establish cause or predict future churn.")

## 5. Additional stakeholder insights

These three outputs address the brief's request for non-obvious ideas beyond theme and sentiment charts. Each is a starting point for review with the owning team, not an automated decision.

In [ ]:
# A. Sales + Support: accounts with combinations of churn, escalation, competitor, or negative-customer flags.
pd.DataFrame(risk)[["account", "risk_score", "meetings", "risk_flags", "mean_customer_sentiment"]].head(10)

In [ ]:
# B. Engineering + Product: themes occurring across multiple call types.
cross_surface = [t for t in graph["cross_surface_themes"]]
cross_surface_labels = {t["theme_id"]: t["label"] for t in themes}
pd.DataFrame({"theme_id": cross_surface, "theme": [cross_surface_labels.get(t, t) for t in cross_surface]})

In [ ]:
# C. Product + Support: candidate bugs, feature requests, process gaps, and praise, grouped by module.
pd.DataFrame(voc)[["voc_type", "module", "count", "why_it_matters"]].sort_values("count", ascending=False).reset_index(drop=True)

### How to use these carefully

- **Risk radar:** score is a rule-based maximum meeting score per inferred account. It is not a probability; inferred account names and phrase matches can be wrong.
- **Cross-surface graph:** a shared primary-theme label links representative meetings of different call types. The implementation does not require the meetings to be within 14 days and does not prove they concern the same incident.
- **VOC backlog:** one transcript may produce a candidate item based on phrases or theme. Counts are not deduplicated product defects or validated feature requests. Read the example quotes before prioritizing.

The React dashboard provides the interactive charts and graph. This notebook is the reproducible, inspectable walkthrough of the same backend outputs.

## 6. Reproducibility checks and next steps

The checks below catch missing call types, empty theme examples, and mismatched summary counts. They verify output shape, not classifier accuracy. Before presenting, manually review at least one quote per theme and explain any weak matches.

In [ ]:
assert len(meetings) == run_result["meetings"], "Pipeline result and processed meeting count differ."
assert set(overview["call_type_mix"]) == {"support", "external", "internal"}, "Expected all three call types."
assert sum(overview["call_type_mix"].values()) == len(meetings), "Call-type counts do not sum to meeting count."
assert all(theme["example_meeting_ids"] for theme in themes), "At least one theme has no example meeting."

high_risk_accounts = {
    m["customer_name"] or m["meeting_id"]
    for m in meetings
    if m["risk_score"] >= 0.5 and m["call_type"] != "internal"
}
assert overview["high_risk_accounts"] == len(high_risk_accounts), "Overview KPI should count distinct accounts."
print("Checks passed.")
print(f"{len(meetings)} meetings | {len(themes)} themes | {len(graph['cross_surface_themes'])} cross-surface themes | {len(voc)} VOC groups")